### IMPORTING LIBRARY

In [1]:
# =============================================
#                   IMPORT
# =============================================
from    preProcessingData                  import PreProcessData
from    utility.matpilotDraw               import MatPlot 
from    utility.plotlyDraw                 import PlotlyDraw
from    tqdm                               import tqdm
from    datetime                           import datetime
from    prettytable                        import PrettyTable
import  plotly.graph_objects                as go
import  numpy                               as np 
import  pandas                              as pd
import  matplotlib.pyplot                   as plt
import  os, warnings,sys
import  random
import  json


np.random.seed(1)
random.seed(1)


# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

loadOriginalFile = False
outputDir       = os.path.expanduser("~/Desktop/Dropbox/codeData/2ndRealDataJournalWithoutGit/")


In [2]:
matPlot    = MatPlot() 
plotly     = PlotlyDraw()
preData    = PreProcessData()

### 1. Data loading

In [3]:
# Use this one only aiming to show the data summary 
if loadOriginalFile:
    crusher1        =   preData.dataLoadingOriginalData(component = "crusher1" )
    crusher2        =   preData.dataLoadingOriginalData(component = "crusher2")
    conveyorBelt    =   preData.dataLoadingOriginalData(component = "conveyorBelt")

    preData.save2H5Format(rawData   = [crusher1,crusher2,conveyorBelt],
                          name      = "originalCrusher",
                          keys       = ["cr1", "cr2" , "cv2"])


In [4]:
if not loadOriginalFile:
    outputDir       = os.path.expanduser("~/Desktop/Dropbox/codeData/2ndRealDataJournalWithoutGit/")
    cr1, cr2, cv2 = preData.loadH5File( dir     =   outputDir + "data/",
                                        name    =   "originalCrusher", 
                                        keys    =  ["cr1", "cr2", "cv2"])

    print(cr1.shape, cr2.shape, cv2.shape)

    crusher1        = cr1
    crusher2        = cr2
    conveyorBelt    = cv2

Process:7919||2025-01-17 08:20:13,671-INFO [preProcessingData.py:loadH5File():569]:Loading H5 file ...
Process:7919||2025-01-17 08:20:16,889-INFO [preProcessingData.py:loadH5File():575]:Finish loading data from files ...


(525399, 41) (525317, 39) (525601, 11)


### 2. Handling with different length

##### 2.1 Handling duplicate in crushers

In [5]:
# Handling duplicate in crushers
print("----------------------- Crusher 1")
duplicates = crusher1[crusher1['tiempo'].duplicated(keep=False)]  # keep=False marks all occurrences as duplicates
print(duplicates)

print("----------------------- Crusher 2")
duplicates = crusher2[crusher2['tiempo'].duplicated(keep=False)]  # keep=False marks all occurrences as duplicates
print(duplicates)

print("----------------------- ConveyorBelt 2")
duplicates = conveyorBelt[conveyorBelt['tiempo'].duplicated(keep=False)]  # keep=False marks all occurrences as duplicates
print(duplicates)

print("=======================================")
print("         HANLDING")
# ---------------------------------
# Remove the duplication
singleCr1            =   crusher1[~crusher1['tiempo'].duplicated(keep='first')]
singleCr2            =   crusher2[~crusher2['tiempo'].duplicated(keep='first')]
singleCv2            =   conveyorBelt[~conveyorBelt['tiempo'].duplicated(keep='first')]

print(singleCr1.shape, singleCr2.shape, singleCv2.shape)

# ---------------------------------
# Test again
# ---------------------------------
print("=======================================")
print("         TEST AGAIN")
print("----------------------- Crusher 1")
duplicates = singleCr1[singleCr1['tiempo'].duplicated(keep=False)]  # keep=False marks all occurrences as duplicates
print(duplicates)

print("----------------------- Crusher 2")
duplicates = singleCr2[singleCr2['tiempo'].duplicated(keep=False)]  # keep=False marks all occurrences as duplicates
print(duplicates)

print("----------------------- ConveyorBelt 2")
duplicates = singleCv2[singleCv2['tiempo'].duplicated(keep=False)]  # keep=False marks all occurrences as duplicates
print(duplicates)


----------------------- Crusher 1
                    tiempo Descripción Equipo Modo de Falla Comentario  \
130979 2023-04-01 23:00:00                NaN           NaN        NaN   
130980 2023-04-01 23:00:00                NaN           NaN        NaN   
130981 2023-04-01 23:01:00                NaN           NaN        NaN   
130982 2023-04-01 23:01:00                NaN           NaN        NaN   
130983 2023-04-01 23:02:00                NaN           NaN        NaN   
...                    ...                ...           ...        ...   
131094 2023-04-01 23:57:00                NaN           NaN        NaN   
131095 2023-04-01 23:58:00                NaN           NaN        NaN   
131096 2023-04-01 23:58:00                NaN           NaN        NaN   
131097 2023-04-01 23:59:00                NaN           NaN        NaN   
131098 2023-04-01 23:59:00                NaN           NaN        NaN   

       Especialidad Motivo Inicio Reportado Fin Reportado  corriente(A)  \
13

##### 2.2 Finding the exception of each matrix (times in crusher1 not include in crusher2 or conveyorBelt)

In [6]:
# Tracking all of them to check which one is exception

crusher1MOF         =   singleCr1['tiempo'].unique()
crusher2MOF         =   singleCr2['tiempo'].unique()
cvMOF               =   singleCv2['tiempo'].unique()

# ---------------------------
# Check the length of each original 
# after filtering duplicatition
# ---------------------------
setCr1 = set(crusher1MOF)
setCr2 = set(crusher2MOF)
setCv2 = set(cvMOF)

print(len(setCr1))
print(len(setCr2))
print(len(setCv2))
# ---------------------------
# Finding the exception
# ---------------------------
onlyCr1 = setCr1 - setCr2
onlyCr2 = setCr2 - setCr1

print("Length of onlyCr1", len(onlyCr1))
print("Only in Crusher 1:")
for idx in onlyCr1:
    print(idx)

print("===========================")
print("Length of onlyCr2", len(onlyCr2))
print("Only in Crusher 2")
for idx in onlyCr2:
    print(idx)

print("===========================")
# ------------------------
# After removing CR1 and CR2,
# Comparing the common result
# with CV2

commonTimes = set(setCr1.intersection(setCr2))
print("commonTimes",len(commonTimes))

onlyCommon  = commonTimes - setCv2
onlyCv2     = setCv2 - commonTimes

print("Length of onlyCommon", len(onlyCommon))
print("Only in onlyCommon")
for idx in onlyCommon:
    print(idx)

print("===========================")
print("Length of onlyCv2", len(onlyCv2))
print("Only in onlyCv2")
for idx in onlyCv2:
    print(idx)

525339
525257
525541
Length of onlyCr1 116
Only in Crusher 1:
2023-07-27T17:18:00.000000000
2023-07-27T16:55:00.000000000
2023-07-27T16:09:00.000000000
2023-07-27T17:36:00.000000000
2023-07-27T17:25:00.000000000
2023-07-27T16:16:00.000000000
2023-07-27T16:46:00.000000000
2023-07-27T17:39:00.000000000
2023-07-27T17:16:00.000000000
2023-07-27T16:07:00.000000000
2023-07-27T16:56:00.000000000
2023-07-27T16:32:00.000000000
2023-07-27T13:09:00.000000000
2023-07-27T16:27:00.000000000
2023-07-27T13:04:00.000000000
2023-07-27T17:20:00.000000000
2023-07-27T17:44:00.000000000
2023-07-27T16:03:00.000000000
2023-07-27T17:15:00.000000000
2023-07-27T17:07:00.000000000
2023-07-27T16:19:00.000000000
2023-07-27T17:23:00.000000000
2023-07-27T13:07:00.000000000
2023-07-27T16:43:00.000000000
2023-07-27T16:59:00.000000000
2023-07-27T16:51:00.000000000
2023-07-27T17:31:00.000000000
2023-07-27T16:41:00.000000000
2023-07-27T17:34:00.000000000
2023-07-27T17:29:00.000000000
2023-07-27T13:08:00.000000000
2023-07-

##### 2.3 Remove redundant time and only hold the joint time from components

In [7]:
commonTimes3Comp    = set(singleCr1['tiempo']).intersection(singleCr2['tiempo'], singleCv2['tiempo'])

# Filter each DataFrame to only keep rows with the common "Time" values
filtedCr1 = singleCr1[singleCr1['tiempo'].isin(commonTimes3Comp)].reset_index(drop=True)
filtedCr2 = singleCr2[singleCr2['tiempo'].isin(commonTimes3Comp)].reset_index(drop=True)
filtedCv2 = singleCv2[singleCv2['tiempo'].isin(commonTimes3Comp)].reset_index(drop=True)

print(filtedCr1.shape,filtedCr2.shape,filtedCv2.shape)

(525223, 41) (525223, 39) (525223, 11)


### 3. Checking different MOF

##### 3.1 Check the intersection

In [8]:
cr1MOF          =   list(filtedCr1['Modo de Falla'].unique())
cr2MOF          =   list(filtedCr2['Modo de Falla'].unique())
cv2MOF          =   list(filtedCv2['Modo de Falla'].unique())

print("--------------------------------")
print("The MOF each components")
print("--------------------------------")
print("Crusher 1", cr1MOF)
print("Crusher 2", cr2MOF)
print("ConveyorBelt",cv2MOF)

print("All MOF",set(cr1MOF + cr2MOF + cv2MOF))
# ------------------------------------
# Check which one is exception MOF
# ------------------------------------
setCr1 = set(cr1MOF)
setCr2 = set(cr2MOF)
setCv2 = set(cv2MOF)
# Elements in list1 but not in list2
only_in_list1 = setCr1 - setCr2

# Elements in list2 but not in list1
only_in_list2 = setCr2 - setCr1

# Elements in both lists
common_elements = setCr1 & setCr2

print("Only in Crusher 1:", only_in_list1)
print("Only in Crusher 2:", only_in_list2)
print("Common elements:", common_elements)

--------------------------------
The MOF each components
--------------------------------
Crusher 1 [nan, 'Stand by', 'Falta de equipo carguío mina', 'Válvula defectuosa', 'Detención por Enclavamiento', 'Sobrecarga, Rebalses y atollos', 'Puesta en Marcha o sistema en vació', 'Colación', 'Falta de inventario', 'Tronadura', 'Stock de producto lleno', 'Falla o problemas aguas abajo', 'Mantención programada', 'Cambio de Turno', 'Otra falla instrumentación', 'Sensor de flujo lectura errónea', 'Falta de Equipos de Apoyo', 'Otra falla mecánica', 'Sensor nivel lectura errónea', 'Bloqueo de Equipos', 'Puesta en marcha de equipos o sistemas en vacío', 'Falla o problemas aguas arriba', 'Limpieza equipo', 'Ajuste de Setting (Asociado al Plan)', 'Falta de aceite hidráulico', 'Factor Climático', 'Corte energía interno', 'Lubricación \xa0deficiente', 'Inspección de equipo ante evento operacional (chatarra, falsa alarma etc)', 'Temperatura fuera de rango', 'Bomba hidráulica con fuga', 'Fuga de aceite'

##### 3.2 Sumarry MOF

In [9]:
preData.sumarryMOF(filtedCr1)
preData.sumarryMOF(filtedCr2)
preData.sumarryMOF(filtedCv2)

+-------------------------------------------------------------------------------------------------------+-----------------------------------+
|                                              Elimination                                              |            Critical MOF           |
+-------------------------------------------------------------------------------------------------------+-----------------------------------+
|                                                Stand by                                               |                nan                |
|                                      Falta de equipo carguío mina                                     |         Válvula defectuosa        |
|                                      Detención por Enclavamiento                                      |       Mantención programada       |
|                                     Sobrecarga, Rebalses y atollos                                    |     Falta de aceite hidráulico    |
|     

##### 3.3 Sumarry the whole data

In [10]:
print("==================================================================================================")
print("                                     CRUSHER1")
markedPoint = preData.sumarryDataFromFile(filtedCr1, printUsefulMOF = True)

print("==================================================================================================")
print("                                     CRUSHER2")
markedPoint = preData.sumarryDataFromFile(filtedCr2, printUsefulMOF = True)

print("==================================================================================================")
print("                                     CONVEYOR BELT")
markedPoint = preData.sumarryDataFromFile(filtedCv2, printUsefulMOF = True)

Process:7919||2025-01-17 08:20:31,219-INFO [preProcessingData.py:sumarryDataFromFile():784]:Sumarry the data


                                     CRUSHER1


Process:7919||2025-01-17 08:20:39,493-INFO [preProcessingData.py:sumarryDataFromFile():784]:Sumarry the data


All the mode of failure: ['Stand by', 'Falta de equipo carguío mina', 'Válvula defectuosa', 'Detención por Enclavamiento', 'Sobrecarga, Rebalses y atollos', 'Puesta en Marcha o sistema en vació', 'Colación', 'Falta de inventario', 'Tronadura', 'Stock de producto lleno', 'Falla o problemas aguas abajo', 'Mantención programada', 'Cambio de Turno', 'Otra falla instrumentación', 'Sensor de flujo lectura errónea', 'Falta de Equipos de Apoyo', 'Otra falla mecánica', 'Sensor nivel lectura errónea', 'Bloqueo de Equipos', 'Puesta en marcha de equipos o sistemas en vacío', 'Falla o problemas aguas arriba', 'Limpieza equipo', 'Ajuste de Setting (Asociado al Plan)', 'Falta de aceite hidráulico', 'Factor Climático', 'Corte energía interno', 'Lubricación \xa0deficiente', 'Inspección de equipo ante evento operacional (chatarra, falsa alarma etc)', 'Temperatura fuera de rango', 'Bomba hidráulica con fuga', 'Fuga de aceite', 'Falla en sistema hidráulico', 'Esperas de personal de reparación y/o repuesto

Process:7919||2025-01-17 08:20:50,722-INFO [preProcessingData.py:sumarryDataFromFile():784]:Sumarry the data


All the mode of failure: ['Stand by', 'Mantención programada', 'Otra falla operacional', 'Puesta en marcha de equipos o sistemas en vacío', 'Otra falla instrumentación', 'Falla o problemas aguas abajo', 'Sensor temperatura lectura errónea', 'Limpieza equipo', 'Otra falla mecánica', 'Falla en sistema hidráulico', 'Factor Externo (huelgas, manifestaciones, cortes de caminos, etc)', 'Inspección de equipo ante evento operacional (chatarra, falsa alarma etc)', 'Otra falla eléctrica', 'Falla o problemas aguas arriba', 'Detención por Enclavamiento', 'Stock de producto lleno', 'Falta de Equipos de Apoyo', 'Corte energía interno', 'Fuga de aceite', 'Puesta en Marcha o sistema en vació', 'Inchancable', 'Correas de transmisión dañadas y/o cortadas', 'Correas transmisión \xa0con deflexión', 'Filtros saturados', 'Esperas de personal de reparación y/o repuestos y/o equipos de apoyo y/o movimiento de traslado (mec)', 'Correas\xa0 transmisión con tensión fuera de rango', 'Toma de muestra', 'Bloqueo de

In [11]:
print(filtedCr2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525223 entries, 0 to 525222
Data columns (total 39 columns):
 #   Column                                         Non-Null Count   Dtype         
---  ------                                         --------------   -----         
 0   tiempo                                         525223 non-null  datetime64[ns]
 1   Descripción Equipo                             140795 non-null  object        
 2   Modo de Falla                                  140795 non-null  object        
 3   Comentario                                     37396 non-null   object        
 4   Especialidad                                   140795 non-null  object        
 5   Motivo                                         140795 non-null  object        
 6   Inicio Reportado                               140795 non-null  datetime64[ns]
 7   Fin Reportado                                  140795 non-null  datetime64[ns]
 8   Corriente(A)                                

### 4. Remove redundant MOF

##### 4.1 Check nan in data

In [12]:
has_nan = filtedCr1.isna().any()
print(has_nan)
print("--------------------------------------")
has_nan = filtedCr2.isna().any()
print(has_nan)
print("--------------------------------------")
has_nan = filtedCv2.isna().any()
print(has_nan)

tiempo                           False
Descripción Equipo                True
Modo de Falla                     True
Comentario                        True
Especialidad                      True
Motivo                            True
Inicio Reportado                  True
Fin Reportado                     True
corriente(A)                     False
velocidad(rpm)                   False
nivel_tolva_compensacion(%)      False
Potencia(kW)                     False
posicion_poste(%)                False
rendimiento(ton)                 False
presion_estanque_aire(bar)       False
Piñón_ext(°C)                    False
Piñón_int(°C)                    False
Piñón_int(mm/s)                  False
Piñón_ext(mm/s)                  False
Eje_piñón_aceite(%)              False
Lubricación_retorno(°C)          False
Lubricación_depósito(°C)         False
Lubricación_entrada(°C)          False
Lubricación_principal(psi)       False
Lubricación_principal(A)         False
Lubricación_respaldo(psi)

##### 4.2 Remove joint non Critical MOF

In [13]:
rmedJointCr1, rmedJointCr2, rmedJointCv2 = preData.removeNonCriticalMOF(df = [filtedCr1,filtedCr2,pd.concat([filtedCv2,filtedCr2[preData.crusher2ConveyorValue]], axis =1 )], mainternanceAsCriticalFailure = True)
print(rmedJointCr1.shape, rmedJointCr2.shape, rmedJointCv2.shape)

(321375, 41) (321375, 39) (321375, 40)


##### 4.3 Remove nonCriticalMOF

In [14]:
# Insert the values of crusher 2 into the conveyor belt as features
print(filtedCr1.shape, filtedCr2.shape, filtedCv2.shape)
cleanCr1    = preData.removeNonCriticalMOF(df = [filtedCr1], mainternanceAsCriticalFailure = True)
cleanCr2    = preData.removeNonCriticalMOF(df = [filtedCr2], mainternanceAsCriticalFailure = True)
cleanCv2    = preData.removeNonCriticalMOF(df = [pd.concat([filtedCv2,filtedCr2[preData.crusher2ConveyorValue]], axis =1 )], mainternanceAsCriticalFailure = True)
print(cleanCr1.shape, cleanCr2.shape, cleanCv2.shape)

(525223, 41) (525223, 39) (525223, 11)
(321375, 41) (466650, 39) (451068, 40)


In [15]:
print("==================================================================================================")
print("                                     CRUSHER1")
markedPoint = preData.sumarryDataFromFile(cleanCr1, printUsefulMOF = True, printMOFTable=True)

print("==================================================================================================")
print("                                     CRUSHER2")
markedPoint = preData.sumarryDataFromFile(cleanCr2, printUsefulMOF = True, printMOFTable=True)

print("==================================================================================================")
print("                                     CONVEYOR BELT")
markedPoint = preData.sumarryDataFromFile(cleanCv2, printUsefulMOF = True, printMOFTable=True)

Process:7919||2025-01-17 08:21:11,803-INFO [preProcessingData.py:sumarryDataFromFile():784]:Sumarry the data


                                     CRUSHER1


Process:7919||2025-01-17 08:21:18,175-INFO [preProcessingData.py:sumarryDataFromFile():784]:Sumarry the data


+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| No |                MOF                |     startingTime    |      endingTime     | staringRow | endingRow |
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| 1  |         Válvula defectuosa        | 2023-01-06 10:52:00 | 2023-01-06 15:13:00 |    4774    |    5035   |
| 2  |       Mantención programada       | 2023-01-19 00:54:00 | 2023-01-19 01:53:00 |   16976    |   17035   |
| 3  |       Mantención programada       | 2023-01-19 03:24:00 | 2023-01-19 23:12:00 |   17126    |   18314   |
| 4  |       Mantención programada       | 2023-02-02 05:28:00 | 2023-02-02 23:53:00 |   27067    |   28172   |
| 5  |       Mantención programada       | 2023-02-21 08:01:00 | 2023-02-22 00:51:00 |   42726    |   43736   |
| 6  |       Mantención programada       | 2023-03-21 00:10:00 | 2023-03-21 01:53:00 |   63050    |   63

Process:7919||2025-01-17 08:21:27,698-INFO [preProcessingData.py:sumarryDataFromFile():784]:Sumarry the data


+----+-------------------------------------------------+---------------------+---------------------+------------+-----------+
| No |                       MOF                       |     startingTime    |      endingTime     | staringRow | endingRow |
+----+-------------------------------------------------+---------------------+---------------------+------------+-----------+
| 1  |              Mantención programada              | 2023-01-04 02:21:00 | 2023-01-05 07:59:00 |    4367    |    6145   |
| 2  |              Mantención programada              | 2023-01-18 08:01:00 | 2023-01-18 18:44:00 |   22235    |   22878   |
| 3  |        Sensor temperatura lectura errónea       | 2023-01-23 22:51:00 | 2023-01-23 23:45:00 |   30066    |   30120   |
| 4  |              Mantención programada              | 2023-01-25 20:00:00 | 2023-01-29 15:59:00 |   30678    |   36197   |
| 5  |           Falla en sistema hidráulico           | 2023-01-30 14:30:00 | 2023-01-30 18:09:00 |   36956    |   37

##### 4.5 Save to h5 file

In [16]:
saveWholeConsistentData = True
if saveWholeConsistentData:
    preData.save2H5Format(dir = outputDir + "data/", rawData = [filtedCr1, filtedCr2, pd.concat([filtedCv2,filtedCr2[preData.crusher2ConveyorValue]], axis =1 )], name = "AdequateConsistentData", keys = ["cr1", "cr2", "cv2"])


saveJointNonCriticalData = True
if saveJointNonCriticalData:
    preData.save2H5Format(dir = outputDir + "data/", rawData = [rmedJointCr1, rmedJointCr2, rmedJointCv2], name = "RemovedJointNonCriticalMOF", keys = ["cr1", "cr2", "cv2"])

saveSeperatedNonCriticalData = True
if saveSeperatedNonCriticalData:
    preData.save2H5Format(dir = outputDir + "data/",rawData = [cleanCr1, cleanCr2, cleanCv2], name = "RemovedWithoutSeperatedNonCriticalMOF", keys = ["cr1", "cr2", "cv2"])

# if saveAllToExcell:
#     finalCr1.to_excel("data/finalCrusher1.xlsx", index=False)
#     finalCr2.to_excel("data/finalCrusher2.xlsx", index=False)
#     finalCv2.to_excel("data/finalConveyor2.xlsx", index=False)

Process:7919||2025-01-17 08:21:37,163-INFO [preProcessingData.py:save2H5Format():545]:Saving data to h5 file ...
/home/ducan/Documents/2ndrealdatajournalchile/preProcessingData.py:547: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['Descripción Equipo', 'Modo de Falla', 'Comentario', 'Especialidad',
       'Motivo', 'Equipo'],
      dtype='object')]

  data.to_hdf(dir + name + ".h5", key=keys[idx], mode= writingMode)
Process:7919||2025-01-17 08:21:37,475-INFO [preProcessingData.py:save2H5Format():549]:Finish saving data cr1 to h5 file ...
Process:7919||2025-01-17 08:21:37,475-INFO [preProcessingData.py:save2H5Format():545]:Saving data to h5 file ...
/home/ducan/Documents/2ndrealdatajournalchile/preProcessingData.py:547: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_